In [1]:
import numpy as np
from scipy.special import jv, jn_zeros
from scipy.sparse.linalg import eigsh, ArpackNoConvergence
from scipy.sparse import diags
from scipy.interpolate import interp1d
from scipy.integrate import quad
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

##############################################################################
# 1. Parámetros Físicos (en nanómetros)
##############################################################################
# Conversión: 1 m = 1e9 nm.
D0 = 2e-9 * 1e18   # 2e-9 m²/s -> 2e9 nm²/s
# D_s = D0*0.05      # 5% de D0
R = 1e-6 * 1e9     # 1e-6 m -> 1000 nm
z_c = 10           # 10 nm (altura crítica)
delta = 2          # 1 nm (ancho de transición)
lambda_param = 0.5 # en nm⁻¹
n_modes_radial = 16
m_max = 16
N_z = 1000         # Número de nodos en z
alpha_grid = 4.0   # Aglomeramiento cerca de z=0
varkappa = 0       # nm/s
L = 50             # nm

##############################################################################
# 2. Función de Difusión Global D(z) (z en nm)
##############################################################################
def D(z_val):
    """Perfil de difusión completo con z en nm.
       D(z) = D0 [exp(-λz)(1-S(z)) + S(z)] donde 
       S(z) = 0.5 [1 + tanh((z-z_c)/δ)].
    """
    S = 0.5 * (1 + np.tanh((z_val - z_c) / delta))
    return D0 * ((1-np.exp(-lambda_param * z_val)) * (1 - S) + S)
    #return D_s+(D0-D_s) * ((1-np.exp(-lambda_param * z_val)) * (1 - S) + S)

##############################################################################
# 3. Generación de la malla en z (nm)
##############################################################################
# j_vals = np.linspace(0, N_z, N_z)
j_vals1 = np.arange(int(N_z*0.6))
j_vals2 = np.arange(int(N_z*0.4))

z1 = z_c * np.sinh(alpha_grid * j_vals1 / (N_z*0.6 - 1)) / np.sinh(alpha_grid) # [0,z_c]
z2 = z_c + (L - z_c) * np.sinh(alpha_grid*j_vals2 / (N_z*0.4 - 1)) / np.sinh(alpha_grid) # [z_c,L]

# z = L * np.sinh(alpha_grid*j_vals / (N_z - 1)) / np.sinh(alpha_grid) # [0,L]
z = np.concatenate((z1[:-2],z2))

D_values = D(z)
# L = np.max(z)  # Extensión axial

plt.figure(figsize=(8,5))
plt.plot(z, D_values/D0, 'b-', lw=1.5)
plt.xlabel('z (nm)')
plt.ylabel('D(z)/D₀')
plt.title('Perfil de Difusión')
plt.grid(ls='--', alpha=0.7)
plt.savefig('Dz_profile.pdf', bbox_inches='tight')
plt.close()

##############################################################################
# 4. Solución Radial R(ρ)
##############################################################################
alpha_zeros = jn_zeros(0, n_modes_radial)
alpha_n = alpha_zeros / R   # R en nm
rho_plot = np.linspace(0.01 * R, 0.99* R, 80)
J0_table = jv(0, alpha_n[:, None] * rho_plot)

# Graficar modos radiales
plt.figure(figsize=(10,6))
for n in range(min(6, n_modes_radial)):
    plt.plot(rho_plot, J0_table[n], label=f"n={n+1}", linewidth=1.5)
plt.xlabel("ρ (nm)", fontsize=12)
plt.ylabel("Amplitud", fontsize=12)
plt.title("Modos Radiales de Bessel", fontsize=14)
plt.legend()
plt.grid(True)
plt.savefig("modos_radiales.pdf", bbox_inches="tight")
plt.close()

##############################################################################
# 5. Solución Axial Z(z)
##############################################################################
def build_system(n):
    try:
        alpha = alpha_n[n]
        h = np.diff(z)
        N = len(z)
        main_diag = np.zeros(N)
        upper_diag = np.zeros(N-1)
        lower_diag = np.zeros(N-1)
        # for j in range(1, N-1):
        #     h_prev = h[j-1]
        #     h_next = h[j]
        #     D_prev = 2 * D_values[j] * D_values[j-1] / (D_values[j] + D_values[j-1] + 1e-30)
        #     D_next = 2 * D_values[j] * D_values[j+1] / (D_values[j] + D_values[j+1] + 1e-30)
        #     h_avg = 0.5 * (h_prev + h_next)
        #     # main_diag[j] = D_prev / (h_prev * h_avg) + D_next / (h_next * h_avg) + alpha**2 * D_values[j]
        #     # upper_diag[j] = -D_next / (h_next * h_avg)
        #     # lower_diag[j-1] = -D_prev / (h_prev * h_avg)
        #     upper_diag[j] = - D_next / h_avg**2
        #     main_diag[j]  = (D_next + D_prev) / h_avg**2 + alpha**2*D_values[j]
        #     lower_diag[j-1] = - D_prev / h_avg**2

        # Índices internos (j=1 a N-2)
        j_internos = np.arange(1, N-1)
        
        # Cálculos vectorizados
        D_prev = 2 * D_values[j_internos] * D_values[j_internos-1] / (D_values[j_internos] + D_values[j_internos-1] + 1e-30)
        D_next = 2 * D_values[j_internos] * D_values[j_internos+1] / (D_values[j_internos] + D_values[j_internos+1] + 1e-30)
        h_avg = 0.5 * (h[j_internos-1] + h[j_internos])
        
        # Asignación a diagonales
        upper_diag[j_internos] = -D_next / h_avg**2
        lower_diag[j_internos-1] = -D_prev / h_avg**2
        main_diag[j_internos] = (D_prev + D_next) / h_avg**2 + alpha**2 * D_values[j_internos]
        
        # Condiciones de frontera:
        # En z=0: superficie reflectante: Z0 = Z1
        main_diag[0]  = D0 + varkappa*h[0]
        upper_diag[0] = -D0
        # En z=L: Neumann: Z(N-1) = Z(N-2)
        main_diag[-1] = 1.0
        lower_diag[-1] = -1.0

        A = diags([main_diag, upper_diag, lower_diag], [0, 1, -1])
        eigenvalues, eigenvectors = eigsh(A, k=m_max, which='SM', tol=1e-6, maxiter=15000)
        return (n, (eigenvalues, eigenvectors))
    except ArpackNoConvergence:
        print(f"Advertencia: Modo axial para n={n} no convergió. Usando ceros.")
        return (n, (np.zeros(m_max), np.zeros((N, m_max))))

axial_results = Parallel(n_jobs=4, verbose=1)(delayed(build_system)(n) for n in range(n_modes_radial))
eigenvalues_ax = {}
eigenvectors_ax = {}
for n, (evals, evecs) in sorted(axial_results, key=lambda x: x[0]):
    eigenvalues_ax[n] = evals
    eigenvectors_ax[n] = evecs

# Normalizar los modos axiales en z
for n in range(n_modes_radial):
    for m in range(m_max):
        norm = np.sqrt(np.trapz(eigenvectors_ax[n][:, m]**2, z))
        if norm != 0:
            eigenvectors_ax[n][:, m] /= norm

# Graficar algunos modos axiales
plt.figure(figsize=(10,6))
for n in range(min(2, n_modes_radial)):
    for m in range(min(2, m_max)):
        plt.plot(z, eigenvectors_ax[n][:, m], label=f"n={n}, m={m}", linewidth=1.5)
plt.xlabel("z (nm)", fontsize=12)
plt.ylabel("Z(z)", fontsize=12)
plt.title("Modos Axiales", fontsize=14)
plt.legend()
plt.grid(True)
plt.savefig("modos_axiales.pdf", bbox_inches="tight")
plt.close()

##############################################################################
# 6. Término Temporal T(t)
##############################################################################
# Ahora, para cada eigenvalor (de la parte axial) usaremos el valor absoluto
# para que T(t)=exp(-|λ| t) decaiga.
def T(t, lam):
    return np.exp(-np.abs(lam) * t)

plt.figure(figsize=(10,6))
t_plot = np.logspace(-12, -5, 100)
# Graficamos T(t) para los modos del sistema axial
for n in range(min(2,n_modes_radial)):
    for m in range(min(2, m_max)):
        lam = eigenvalues_ax[n][m]
        plt.loglog(t_plot, T(t_plot, lam), label=f"n={n}, m={m}", linewidth=1.5)
plt.xlabel("Tiempo (s)", fontsize=12)
plt.ylabel("T(t)", fontsize=12)
plt.title("Decaimiento Temporal T(t)", fontsize=14)
plt.legend()
plt.grid(True, which="both")
plt.savefig("T_t.pdf", bbox_inches="tight")
plt.close()

##############################################################################
# 7. Cálculo de Coeficientes A_{nm}
##############################################################################
# Se asume que la condición inicial es:
#     P(ρ, z, 0) ∝ δ(ρ-ρ0) δ(z-z0) / (2πρ0)
# Con ρ0 = 750 nm y z0 = 10 nm.
rho0 = 750  # nm
z0 = 10     # nm

# Creamos funciones de interpolación para los modos axiales (resultado de la parte axial)
Z_interps = {}
for n in range(n_modes_radial):
    Z_interps[n] = [interp1d(z, eigenvectors_ax[n][:, m], bounds_error=False, fill_value=0.0) for m in range(m_max)]

def compute_A_nm(n):
    # Calcula la norma en la dirección z para el modo m
    Z_norms = np.array([quad(lambda z_val: Z_interps[n][m](z_val)**2 / D(z_val), z[0], z[-1], limit=300)[0] for m in range(m_max)])
    # Evalúa cada modo en z = 15 nm
    Z_m_z0 = np.array([Z_interps[n][m](z0) for m in range(m_max)])
    # Evalúa el modo radial en ρ0 = 750 nm
    J0_rho0 = jv(0, alpha_n[n] * rho0)
    valid = (Z_norms > 1e-25) & (np.abs(Z_m_z0) > 1e-12)
    return (J0_rho0 * Z_m_z0 * valid) / (2 * np.pi * rho0 * (Z_norms + 1e-30))

A_nm = Parallel(n_jobs=4)(delayed(compute_A_nm)(n) for n in range(n_modes_radial))

##############################################################################
# 8. Densidad de Probabilidad P(ρ,z,t)
##############################################################################
def compute_P(rho_vals, z_vals, t):
    P = np.zeros((len(z_vals), len(rho_vals)))
    # Para cada modo se usa T_{n,m}(t)= exp(-|λ_{n,m}|t)
    for i, rho in enumerate(rho_vals):
        J0_vals = jv(0, alpha_n * rho)
        for j, z_val in enumerate(z_vals):
            s = 0.0
            for n in range(n_modes_radial):
                for m in range(m_max):
                    Z_val = Z_interps[n][m](z_val)
                    T_val = np.exp(np.clip(-np.abs(eigenvalues_ax[n][m]) * t, -700, 700))
                    s += A_nm[n][m] * J0_vals[n] * Z_val * T_val
            P[j, i] = np.abs(s)
    return P

rho_vals = np.linspace(0.01 * R, 0.99 * R, 50)
z_vals = np.linspace(0.01*L, 0.99*L, 50)
for t_val in [1e-12,1e-11,1e-10,1e-9,1e-8,1e-7,1e-6,1e-5,1e-4,1e-3]:
    P_val = compute_P(rho_vals, z_vals, t=t_val)
    
    plt.figure(figsize=(10,8))
    plt.contourf(rho_vals, z_vals, P_val, 50, cmap="plasma")
    plt.xlabel("ρ (nm)", fontsize=12)
    plt.ylabel("z (nm)", fontsize=12)
    plt.title(f"Densidad P(ρ,z,t) para t = {t_val}", fontsize=14)
    plt.colorbar()
    plt.savefig(f"P_contour_{t_val}.pdf", bbox_inches="tight")
    plt.close()

##############################################################################
# 9. Función de Correlación G(t)
##############################################################################
def compute_G(t, N_quad=40):
    # Dominios en nm: ρ en [0.3R, 0.7R] y z en [9,11] nm.
    z_min = 7
    z_max = 13
    j_nodes = np.linspace(z_min, z_max, N_quad)
    
    # Mallado no uniforme en z
    z_nodes = z_min + (z_max - z_min) * np.sinh(alpha_grid*j_nodes / (N_quad - 1))/np.sinh(alpha_grid) # [z_min,z_max]

    # calculamos pesos en z con la regla del trapecio no uniforme
    dz = np.diff(z_nodes)
    wz = np.empty_like(z_nodes)
    wz[0]    = dz[0]/2
    wz[-1]   = dz[-1]/2
    wz[1:-1] = 0.5*(dz[:-1]+dz[1:])

    # Mallado uniforme en rho
    rho_nodes = np.linspace(0.1 * R, 0.9 * R, N_quad)
    total = 0.0
    for rho in rho_nodes:
        J0_vals = jv(0, alpha_n * rho)
        for j,z_val in enumerate(z_nodes):
            f = (3 * z_val**2 - rho**2) / (rho**2 + z_val**2 + 1e-30)**2.5
            s = 0.0
            for n in range(n_modes_radial):
                for m in range(m_max):
                    Z_val = Z_interps[n][m](z_val)
                    T_val = np.exp(np.clip(-np.abs(eigenvalues_ax[n][m]) * t, -700, 700))
                    s += A_nm[n][m] * J0_vals[n] * Z_val * T_val
            total += np.abs(rho*f*s*wz[j])
    dr = (0.9 * R - 0.1 * R) / N_quad
    result = (5 / (16 * np.pi)) * (total * dr)
    return result

t_values = np.logspace(-12, -5, 50)
G_values = Parallel(n_jobs=4)(delayed(compute_G)(t) for t in t_values)

np.savetxt("G_vs_t.dat", np.c_[t_values, G_values], header="Tiempo(s) G(t)", fmt="%.3e")

plt.figure(figsize=(10,6))
plt.loglog(t_values, G_values, "o-", label="Simulación", color="navy", markersize=6, linewidth=1.5)
plt.plot(t_values, 1e-8 * t_values**-1.5, "--", label=r"Teoría $t^{-3/2}$", color="crimson", linewidth=1.5)
plt.xlabel("Tiempo (s)", fontsize=12)
plt.ylabel("G(t)", fontsize=12)
plt.title("Función de Correlación Nano-NMR", fontsize=14)
plt.legend()
plt.grid(which="both", alpha=0.4)
plt.savefig("correlacion_nanoNMR.pdf", bbox_inches="tight", dpi=300)
plt.close()

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  16 out of  16 | elapsed:  1.2min finished
